# Look, It's a Fake! — Part I

**Competition**: UB Master FDS — Kaggle 2025/26 (Part I)
**Team**: Frenchies — Clara Bouvier, Bastien Laplace
**Result**: 3rd / 17 (tied with 2nd)
**Metric**: Accuracy

## Problem

Detect LLM-generated content across two heterogeneous datasets:
- **Dataset A** (`derma`): tabular health/dermatology data — numeric and binary features with heavy missing values.
- **Dataset B** (`text`): news headlines — short free-text, binary label (`fake` / `real`).

## Hard Constraint

Only `LogisticRegression(penalty=None)` is allowed. No ensembles, no regularization, no kernel tricks. Every accuracy gain must come entirely from feature engineering and preprocessing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Dependencies

Standard scientific Python stack. Two tools are central:
- `KBinsDiscretizer`: allows the linear model to learn non-monotonic threshold effects by converting continuous features into one-hot bin indicators.
- `TfidfVectorizer` / `CountVectorizer`: turn raw text into sparse feature matrices that a logistic classifier can operate on directly.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler



## Data Loading

Five files: train/test splits for each dataset plus a sample submission template. Loaded directly from Google Drive (Colab environment).

In [3]:
data_example = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-i/example.csv')
data_test_a_derma = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-i/test_A_derma.csv')
data_test_b_text = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-i/test_B_text.csv')
data_train_a_derma = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-i/train_A_derma.csv')
data_train_b_text = pd.read_csv(r'/content/drive/MyDrive/Kaggle_competition_its_a_fake/look-it-is-a-fake-2526-part-i/train_B_text.csv')

---
## Dataset A — Tabular Dermatology Data

The dermatology dataset has numeric features, binary indicator columns, and significant missing values. Target: binary `fake` vs `real`.

### Preprocessing strategy

`LogisticRegression(penalty=None)` is unregularized, so the model is sensitive to feature scale and irrelevant columns. The pipeline addresses six issues:

1. **Leaky feature removed** — `Doughnuts consumption` was identified as a data-leakage artifact and dropped entirely.
2. **Logical imputation for mutually exclusive groups** — size columns (`Small size`, `Mid size`, `Large size`) and area columns are mutually exclusive indicators: if one is 1, the others must be 0, not NaN. Row-wise logic fills these before any statistical imputation runs.
3. **Discretization of continuous features** — `Genetic Propensity` and `Skin X test` are binned via quantile-based `KBinsDiscretizer` (5 bins, one-hot encoded). This lets the linear classifier approximate non-monotonic relationships it otherwise cannot express.
4. **Interaction features** — `Propensity × X test`, a size/area ratio proxy, and `Large_Coexistence` capture co-occurrence effects invisible to a model that only sees individual features.
5. **Missing indicators + mean imputation** — instead of silently imputing, a binary flag is added for every feature that has missing values. Missingness itself can be predictive, and the flag preserves that signal after imputation.
6. **RobustScaler** — chosen over StandardScaler because it uses median and IQR rather than mean and std, making it insensitive to the outliers present in this dataset.

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

LEAKY_FEATURE = 'Doughnuts consumption'
TARGET_COLUMN = 'Fake/Real'
ID_COLUMN = 'Id'
NAN_FILL_VALUE_LESSION = 0.0

SIZE_COLS = ['Small size', 'Mid size', 'Large size']
AREA_COLS = ['Mid', 'Small', 'Large']
DISCRETIZE_COLS = ['Genetic Propensity', 'Skin X test']
N_BINS = 5

def apply_discretization(df, cols, n_bins, mode='train', discretizer=None):
    if mode == 'train':
        discretizer = KBinsDiscretizer(n_bins=n_bins, encode='onehot-dense', strategy='quantile', random_state=42)
        X_binned = discretizer.fit_transform(df[cols])
    elif mode == 'test':
        X_binned = discretizer.transform(df[cols])

    actual_bin_names = []
    discretizer_to_use = discretizer

    if hasattr(discretizer_to_use, 'n_bins_'):
        for col_name, actual_bins in zip(cols, discretizer_to_use.n_bins_):
            actual_bin_names.extend([f'{col_name}_bin_{i}' for i in range(actual_bins)])
    else:
        actual_bin_names = [f'binned_feature_{i}' for i in range(X_binned.shape[1])]

    df_discretized = pd.DataFrame(
        X_binned,
        columns=actual_bin_names,
        index=df.index
    )

    df = df.drop(columns=cols, errors='ignore')
    df = pd.concat([df, df_discretized], axis=1)

    return df, discretizer


def preprocess_dataset_a_v3(df_raw, mode='train', train_imputer_values=None, train_nan_cols=None, discretizer_obj=None, discretize_imputer_values=None):
    df = df_raw.copy()

    df_ids = df[ID_COLUMN] if ID_COLUMN in df.columns else None
    df = df.drop(columns=[ID_COLUMN, LEAKY_FEATURE], errors='ignore')

    if 'Lession' in df.columns:
        df['Lession'] = df['Lession'].fillna(NAN_FILL_VALUE_LESSION)

    for group in [SIZE_COLS, AREA_COLS]:
        for index, row in df.iterrows():
            if row[group].max() == 1.0:
                df.loc[index, group] = row[group].fillna(0.0)

    df['Propensity_X_Test'] = df['Genetic Propensity'] * df['Skin X test']
    df['Size_Area_Ratio_Proxy'] = (df['Small size'] + df['Mid size']) / (df['Small'] + df['Mid'] + 1e-6)
    df['Large_Coexistence'] = df['Large size'] * df['Large']

    if mode == 'train':
        discretize_imputer_values = df[DISCRETIZE_COLS].mean()
        df[DISCRETIZE_COLS] = df[DISCRETIZE_COLS].fillna(discretize_imputer_values)
    elif mode == 'test':
        df[DISCRETIZE_COLS] = df[DISCRETIZE_COLS].fillna(discretize_imputer_values)

    if mode == 'train':
        df, discretizer_obj = apply_discretization(df, DISCRETIZE_COLS, N_BINS, mode='train')
    elif mode == 'test':
        df, _ = apply_discretization(df, DISCRETIZE_COLS, N_BINS, mode='test', discretizer=discretizer_obj)

    if mode == 'train':
        current_nan_cols = df.columns[df.isna().any()].tolist()
    else:
        current_nan_cols = train_nan_cols

    for col in current_nan_cols:
        indicator_name = f'{col}_is_missing'
        if col in df.columns:
            df[indicator_name] = df[col].isna().astype(int)

    if mode == 'train':
        final_imputation_values = df.mean()
        df_processed = df.fillna(final_imputation_values)
        return df_processed, final_imputation_values, current_nan_cols, df_ids, discretizer_obj, discretize_imputer_values

    elif mode == 'test':
        df_processed = df.fillna(train_imputer_values)
        feature_cols_train = train_imputer_values.index.tolist()
        df_processed = df_processed.reindex(columns=feature_cols_train, fill_value=0.0)
        return df_processed, df_ids

def map_target(y_series):
    return y_series.map({'fake': 0, 'real': 1}).astype(int)


### Training — Dataset A

The preprocessing and scaler are fitted **only on training data** and applied to the test set — this includes the discretization imputer values — to prevent any form of data leakage.

An 80/20 stratified hold-out provides an honest pre-submission accuracy estimate before generating test predictions.

In [ ]:
yA = map_target(data_train_a_derma[TARGET_COLUMN])
XA = data_train_a_derma.drop(columns=[TARGET_COLUMN], errors='ignore')

XA_processed, imputer_values, nan_cols, _, discretizer_obj, discretize_imputer_values = preprocess_dataset_a_v3(XA, mode='train')
X_train_full = XA_processed
y_train_full = yA

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)
feature_columns = X_train.columns.tolist()

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

modelA = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=2000,
    random_state=42
)
modelA.fit(X_train_scaled, y_train)

y_pred_val = modelA.predict(X_val_scaled)
acc_val = accuracy_score(y_val, y_pred_val)

print("--- Validation Results ---")
print(f"[Hold-out 80/20] Accuracy = {acc_val:.4f}")
print("\nClassification report:\n",
      classification_report(y_val, y_pred_val, target_names=["fake","real"]))

X_test_processed, test_ids = preprocess_dataset_a_v3(
    data_test_a_derma,
    mode='test',
    train_imputer_values=imputer_values,
    train_nan_cols=nan_cols,
    discretizer_obj=discretizer_obj,
    discretize_imputer_values=discretize_imputer_values
)

X_test_scaled = scaler.transform(X_test_processed)
y_test_pred_numeric = modelA.predict(X_test_scaled)

label_map_inv = {0: 'fake', 1: 'real'}

submission_df_int_disc_feat_bin5 = pd.DataFrame({
    'Id': test_ids.values,
    'Fake/Real': pd.Series(y_test_pred_numeric).map(label_map_inv).values
})

print("\n--- Test Set Predictions ---")
print(submission_df_int_disc_feat_bin5.head())
print(f"\nSubmission shape: {submission_df_int_disc_feat_bin5.shape}")


---
## Dataset B — News Headlines (Text)

Short news headlines, binary label `fake` vs `real`. A linear model must rely entirely on the feature representation — no depth, no attention, no embeddings.

### Feature engineering strategy

Four complementary feature sets are combined via `ColumnTransformer`:

1. **Word-level TF-IDF** (unigrams–4-grams, 20k features, sublinear TF): captures lexical n-gram patterns. 4-grams pick up short fixed phrases characteristic of LLM-generated text.
2. **Character-level CountVectorizer** (2–4 grams, 20k): captures sub-word patterns and morphology — useful for detecting unnatural word constructions.
3. **Subword TF-IDF** (`char_wb`, 3–5 grams): boundary-aware character n-grams that complement (2) by respecting word edges.
4. **Granular style features** (hand-crafted): capitalization rate, punctuation density, quote presence, word count, unique-word ratio, average word length, and keyword flags (clickbait, sensationalist vocabulary, video references). These target the stylistic fingerprint of LLM-generated headlines vs. real journalism.

The motivation for combining all four: LLM-generated text differs from human text on **multiple axes simultaneously** — lexical choices, sub-word structure, and surface style — and no single representation captures all of them.

`max_iter` is tuned via 5-fold cross-validation grid search over `{200, 400, 600, 700, 800}`. With `penalty=None` the SAGA solver's convergence trajectory acts as soft early stopping — searching over iteration counts finds the point where the model generalizes best before overfitting the training distribution.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from scipy.sparse import csr_matrix
from sklearn.metrics import accuracy_score

train_df = data_train_b_text.copy()
test_df = data_test_b_text.copy()

train_df.columns = ['Id', 'Title', 'Label']
test_df.columns = ['Id', 'Title']
train_df["Title"] = train_df["Title"].fillna("")
test_df["Title"] = test_df["Title"].fillna("")
train_df["Label"] = train_df["Label"].map({"real": 0, "fake": 1})

X_full = train_df[['Title']]
y_full = train_df["Label"]
X_test_data = test_df[['Title']]
test_ids = test_df['Id']


class GranularFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        titles = pd.Series(np.ravel(X)).astype(str)
        features_list = []

        sketchy_flag_words = ['warn', 'alert', 'secret', 'exposed', 'lie', 'fraud', 'scandal', 'shocking', 'unbelievable', 'outrage']
        clickbait_flag_words = ['must', 'insane', 'savage', 'read']
        video_flag_words = ['watch', 'video', 'youtube']

        for text in titles:
            text_lower = text.lower()
            list_of_words = re.findall(r'\w+', text_lower)
            word_count = len(list_of_words)

            num_words = len(text.split())
            text_length = len(text)
            num_capitals = sum(1 for i, c in enumerate(text) if c.isupper() and i > 0)
            punct_count = text.count('!') + text.count('?')
            quote_present = 1 if ('"' in text or "'" in text) else 0
            numerical_tokens = len(re.findall(r'\d+', text))
            unique_word_ratio = len(set(list_of_words)) / word_count if word_count > 0 else 0
            avg_word_length = sum(len(word) for word in list_of_words) / word_count if word_count > 0 else 0

            specific_flags = [1 if word in text_lower else 0 for word in sketchy_flag_words + clickbait_flag_words]
            video_flag = 1 if any(word in text_lower for word in video_flag_words) else 0

            features_list.append([
                num_words, text_length, num_capitals, punct_count, quote_present,
                numerical_tokens, unique_word_ratio, avg_word_length, video_flag
            ] + specific_flags)

        return np.array(features_list, dtype=float)


style_feature_pipeline = Pipeline([
    ("extractor", GranularFeatureTransformer()),
    ("scaler", StandardScaler(with_mean=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("word",         TfidfVectorizer(max_features=20000, ngram_range=(1, 4), stop_words=list(ENGLISH_STOP_WORDS), min_df=2, max_df=0.95, sublinear_tf=True), "Title"),
        ("char",         CountVectorizer(analyzer="char", ngram_range=(2, 4), min_df=2, max_features=20000, lowercase=False), "Title"),
        ("subword_tfidf",TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=20000, sublinear_tf=True), "Title"),
        ("numeric_flags",style_feature_pipeline, "Title")
    ],
    remainder="drop"
)

classifier_tune = LogisticRegression(
    penalty=None, solver="saga", random_state=42, n_jobs=-1,
    max_iter=5000
)

ensemble_pipeline_ct = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", classifier_tune)
])

param_grid_iter = {
    'clf__max_iter': [200, 400, 600, 700, 800]
}

grid_search_iter = GridSearchCV(
    ensemble_pipeline_ct,
    param_grid_iter,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

print("--- Starting max_iter optimization ---")
grid_search_iter.fit(X_full, y_full)

best_iter = grid_search_iter.best_params_['clf__max_iter']
best_score = grid_search_iter.best_score_

classifier_final = LogisticRegression(
    penalty=None, solver="saga", max_iter=best_iter, random_state=42, n_jobs=-1
)

final_submission_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", classifier_final)
])

final_submission_pipeline.fit(X_full, y_full)

y_test_pred = final_submission_pipeline.predict(X_test_data)
pred_labels = pd.Series(y_test_pred).map({0: "real", 1: "fake"})

submission_df_B_final_max_iter = pd.DataFrame({
    "Id": test_ids,
    "Fake/Real": pred_labels
})

print("\n" + "="*80)
print(f"Best CV accuracy: {best_score:.4f}  |  Optimal max_iter: {best_iter}")
print("="*80)
print(submission_df_B_final_max_iter.head())


---
## Submission

Dataset A and Dataset B predictions are generated independently and must be merged into a single submission file. Dataset B IDs need to be offset by the size of Dataset A's test set to avoid ID collisions.

In [7]:
def merge_submission_dfs(df_a, df_b, sample_submission_path=None, output_path="submission_final.csv"):
    for df_name, df in zip(["A", "B"], [df_a, df_b]):
        if "Fake/Real" not in df.columns:
            raise ValueError(f"DataFrame {df_name} is missing column 'Fake/Real'.")
        if "Id" not in df.columns:
            raise ValueError(f"DataFrame {df_name} is missing column 'Id'.")

    offset = df_a["Id"].max() + 1
    df_b = df_b.copy()
    df_b["Id"] = df_b["Id"] + offset

    submission_final = pd.concat([df_a, df_b], ignore_index=True)
    submission_final = submission_final.sort_values(by="Id").reset_index(drop=True)

    if sample_submission_path is not None:
        try:
            sample = pd.read_csv(sample_submission_path)
            expected_rows = len(sample)
            if len(submission_final) != expected_rows:
                print(f"Warning: {len(submission_final)} rows generated, {expected_rows} expected.")
            else:
                print("Row count matches sample_submission.csv")
        except Exception as e:
            print(f"Could not read {sample_submission_path}: {e}")

    submission_final.to_csv(output_path, index=False)
    print(f"Merged: {len(df_a)} (A) + {len(df_b)} (B) = {len(submission_final)} rows")
    print(f"Saved: {output_path}")

    return submission_final


### Generate Final Submission File

Concatenate both prediction DataFrames. The ID offset is computed automatically from the maximum ID in Dataset A's output.

In [ ]:
final_submission = merge_submission_dfs(
    submission_df_int_disc_feat_bin5,
    submission_df_B_final_max_iter,
    sample_submission_path="sample_submission_df_A_median_Standard_Scaler_submission_df_B_tfidf.csv"
)

---
## Results & Takeaways

**Final leaderboard**: 3rd / 17 (tied with 2nd)

| Dataset | Validation Accuracy |
|---|---|
| A — Dermatology (tabular) | 0.8083 |
| B — News Headlines (text) | 0.8538 |

### Decisions that mattered

- **Dropping the leaky feature** (`Doughnuts consumption`) was the single most important step for Dataset A. Keeping it would inflate training accuracy while hurting test performance.
- **Discretization over raw continuous inputs**: binning `Genetic Propensity` and `Skin X test` allowed the linear model to capture threshold effects it cannot express with continuous values alone.
- **Missing indicators alongside imputation**: treating missingness as an explicit signal rather than noise improved accuracy on Dataset A.
- **Multi-view text representation**: combining word n-grams, character n-grams, and hand-crafted style flags outperformed any single representation. LLM-generated text has a detectable stylistic signature even in very short headlines.
- **`max_iter` as implicit regularization**: with `penalty=None`, tuning `max_iter` via CV is the only available lever to control overfitting in the SAGA solver. The optimal value (800) indicated full convergence was beneficial — not early stopping.